# Customer Engagement — Regression & Classification Baselines

Artificial Intelligence (M40651) · Week 2

Predicting monthly spend and subscription renewal from five behavioural features.

**Upload `customer_engagement_practice.csv` using the folder icon on the left before running.**

## Task 1 — Load and inspect

In [ ]:
import pandas as pd

df = pd.read_csv('/content/customer_engagement_practice.csv')

print(df.head())
print('Shape:', df.shape)
df.info()

print('\nMissing values:')
print(df.isnull().sum())

print('\nClass counts for renewed_subscription:')
print(df['renewed_subscription'].value_counts())

180 rows, 8 columns. No missing values. 96 customers did not renew, 84 did — close enough to even that accuracy will be a fair measure later.

`customer_id` is excluded from the features: it is a reference number rather than a behaviour, and because every value is unique a model could memorise individual IDs instead of learning anything.

## Task 2 — Linear Regression (predicting `monthly_spend`)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X = df[['website_visits', 'minutes_on_site', 'emails_clicked',
        'previous_spend', 'support_tickets']]
y = df['monthly_spend']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred.round(1)
})
print(results.head(10))

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('MAE:', round(mae, 2))
print('R-squared:', round(r2, 2))

**MAE £15.17 · R² 0.69**

The model is typically about £15 away from what a customer actually spent. R² of 0.69 means it explains roughly 69% of the variation that guessing the average would miss — it is not a percentage of correct answers, because in regression nothing is simply right or wrong.

In [ ]:
# what the model learned
for name, coef in zip(X.columns, model.coef_):
    print(f'{name:<18} {coef:+.2f}')
print(f'{"intercept":<18} {model.intercept_:+.2f}')

Support tickets is the only negative coefficient — more complaints, less spend.

## Task 3 — Logistic Regression (predicting `renewed_subscription`)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score

X = df[['website_visits', 'minutes_on_site', 'emails_clicked',
        'previous_spend', 'support_tickets']]
y = df['renewed_subscription']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print('Test accuracy:', round(accuracy, 3))
print('Confusion matrix:')
print(cm)   # rows = what really happened, columns = what the model said

**Test accuracy 80.6%** — 29 of 36 correct.

18 customers who left were correctly identified, and 11 who stayed. Of the 7 errors, 6 were false alarms (predicted to leave, actually stayed) and 1 was a miss (predicted to stay, actually left). Those two mistakes cost a business very different amounts, which is why the matrix matters more than the single accuracy figure.

In [ ]:
# cross-validation runs on the training data only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print('Cross-validation scores:', cv_scores.round(3))
print('Mean cross-validation accuracy:', round(cv_scores.mean(), 3))

## The finding

One split said **80.6%**. Five folds said **70.2%**, with individual scores from **0.552 to 0.821** — a spread of nearly 27 points depending purely on which customers landed in the test set.

The 80.6% was a favourable split. Stopping there would have meant reporting a model about ten points better than it actually is.

The spread is as informative as the mean: with only 180 rows this model is not stable, and a single train/test split would have hidden that completely.